# JobCheck — EMSCAD Exploratory Data Analysis (EDA)

This notebook checks the EMSCAD job-posting dataset before rule-based fraud detection.

**Expected CSV path:** `data/raw/emscad.csv` relative to the project root. If your filename or path differs, update `DATA_PATH` in the first code cell.

The notebook does not train or tune the detector; it only explores the data.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# This notebook is expected to be saved in job_check/notebooks/eda.ipynb.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'emscad.csv'
print('Project root:', PROJECT_ROOT)
print('Dataset path:', DATA_PATH)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Dataset not found at {DATA_PATH}. Put the CSV at data/raw/emscad.csv '
        'or update DATA_PATH to the actual location.'
    )

df = pd.read_csv(DATA_PATH, low_memory=False)
print('Dataset shape:', df.shape)
display(df.head())

## 1. Column names and data types

In [ ]:
display(pd.DataFrame({
    'column': df.columns,
    'dtype': [str(dtype) for dtype in df.dtypes],
    'missing_values': [int(df[col].isna().sum()) for col in df.columns],
    'missing_percent': [round(df[col].isna().mean() * 100, 2) for col in df.columns],
}))

## 2. Missing values and duplicate rows

In [ ]:
print('Duplicate rows:', int(df.duplicated().sum()))
print('Rows with any missing value:', int(df.isna().any(axis=1).sum()))
print('\nColumns with most missing values:')
display(df.isna().sum().sort_values(ascending=False).head(15).to_frame('missing_count'))

## 3. Genuine vs fraudulent postings

In [ ]:
LABEL_COL = 'fraudulent'
if LABEL_COL not in df.columns:
    raise KeyError(f"Expected label column '{LABEL_COL}'. Available columns: {list(df.columns)}")

labels = pd.to_numeric(df[LABEL_COL], errors='coerce')
print('Missing/invalid labels:', int(labels.isna().sum()))
print('Label counts (0=genuine, 1=fraudulent):')
display(labels.value_counts(dropna=False).sort_index().to_frame('count'))

valid_labels = labels.dropna().astype(int)
ax = valid_labels.map({0: 'Genuine (0)', 1: 'Fraudulent (1)'}).value_counts().plot(
    kind='bar', rot=0, title='Genuine vs Fraudulent Job Postings', ylabel='Number of postings'
)
plt.tight_layout()
plt.show()

## 4. Job-description text length (if available)

In [ ]:
text_candidates = ['description', 'requirements', 'benefits']
available_text_cols = [col for col in text_candidates if col in df.columns]
if not available_text_cols:
    print('No standard text columns found. Available columns:', list(df.columns))
else:
    for col in available_text_cols:
        df[f'{col}_char_count'] = df[col].fillna('').astype(str).str.len()
    chosen_col = 'description_char_count' if 'description_char_count' in df.columns else f'{available_text_cols[0]}_char_count'
    if LABEL_COL in df.columns:
        df.boxplot(column=chosen_col, by=LABEL_COL, grid=False)
        plt.title(f'{chosen_col} by fraud label')
        plt.suptitle('')
        plt.xlabel('fraudulent (0=genuine, 1=fraudulent)')
        plt.ylabel('Character count')
        plt.tight_layout()
        plt.show()
    display(df[available_text_cols].fillna('').astype(str).agg(lambda s: s.str.len().describe()).T)


## 5. Short findings template

After running the notebook, write 2–4 evidence-based observations here. For example:

- How many rows and columns are present?
- Is the label distribution imbalanced?
- Which columns have the most missing values?
- Do description lengths appear different across the two labels? Avoid claiming this proves fraud.